# Lab 11 — Multiple & Polynomial Regression

In this lab you will extend linear regression to multiple features, learn to diagnose multicollinearity with the Variance Inflation Factor, and use polynomial features to fit curves while guarding against overfitting with cross-validation.

The exercises below reuse the same sklearn pipeline pattern from your last lab: split, fit, predict, evaluate. You will see how naturally it extends to more features and to engineered features.

**Concepts covered:** multiple linear regression, coefficient interpretation, multicollinearity, Variance Inflation Factor (VIF), polynomial features, overfitting, cross-validation for hyperparameter selection.

**Reference working sessions:**
- `working-sessions/supervised/02_multiple_linear_regression.ipynb`
- `working-sessions/supervised/03_polynomial_regression.ipynb`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_california_housing,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'C',
    'q3': 'A',
    'q4': 'C',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [2]:
# Q1 — In a multiple linear regression model predicting home price from
# square footage and number of bedrooms, the coefficient for square footage
# is 120. Holding number of bedrooms constant, what does this coefficient mean?
#
#   A) Each additional bedroom increases price by $120
#   B) Each additional square foot is associated with a $120 increase in
#      predicted price, holding bedrooms constant
#   C) The model is 120% confident in the square footage relationship
#   D) Square footage explains 120% of the variance in price

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/02_multiple_linear_regression.ipynb. " \
    "A coefficient in a multiple regression model is a partial effect: it holds all other features fixed."
print("✓ Question 1 correct!")

AssertionError: Don't forget to fill in your answer!

In [ ]:
# Q2 — A feature in your regression model has a VIF of 14. What does this tell you?
#
#   A) The feature explains 14% of the variance in the target
#   B) The feature is uncorrelated with all other features in the model
#   C) The feature's coefficient variance is severely inflated due to strong
#      correlation with other features, so its estimate should not be trusted
#      at face value
#   D) The model's R² will drop by 14% if this feature is removed

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/02_multiple_linear_regression.ipynb. " \
    "Look at the Variance Inflation Factor section and the VIF severity thresholds."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Polynomial regression can fit curves, yet it is still considered a
# "linear model." Why?
#
#   A) The model is linear in its coefficients — it fits a linear combination
#      of transformed features (x, x^2, x^3, ...), even though the resulting
#      curve is nonlinear in x
#   B) The features are transformed onto a linear scale using StandardScaler
#      before fitting
#   C) The predictions form a straight line when plotted against the
#      polynomial degree
#   D) It only uses addition and no multiplication in the model equation

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/supervised/03_polynomial_regression.ipynb. " \
    "The normal equations only require the model to be linear in the coefficients, not the raw feature."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — You are choosing the polynomial degree for a model. What is the
# correct way to select it?
#
#   A) Pick the degree that minimizes error on the training set — lower
#      training error always means a better model
#   B) Always use the highest degree available, since more flexibility can
#      only help
#   C) Use cross-validation to compare validation performance across degrees,
#      and pick the degree that generalizes best, not the one that fits
#      training data most closely
#   D) Pick degree 1, since higher degrees are never worth the added complexity

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/supervised/03_polynomial_regression.ipynb. " \
    "Training error always improves with degree — validation performance is what tells you when to stop."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: Interpreting a multiple regression coefficient"),
    (q2_answer, _ak['q2'], "Q2: Reading a Variance Inflation Factor value"),
    (q3_answer, _ak['q3'], "Q3: Why polynomial regression is still a linear model"),
    (q4_answer, _ak['q4'], "Q4: Selecting polynomial degree with cross-validation"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below extend the sklearn pipeline pattern to multiple features, add a multicollinearity check, and show polynomial regression's overfitting risk directly. Run them in order, each step builds on the previous one.

Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
X, y = load_california_housing()

print("Dataset shape:", X.shape)
print("Features:", X.columns.tolist())
print(f"Target range: ${y.min():.2f}  to  ${y.max():.2f}  (in $100k units)")

### B1 — Fit multiple regression and rank coefficients

With all 8 features in the model at once, each coefficient represents the effect of that feature holding every other feature constant. Fill in the blanks to fit the model and rank the coefficients by absolute size.

In [ ]:
# B1 — Fit multiple linear regression and rank coefficients by absolute value

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=___, random_state=___   # YOUR CODE — hold out 20% of the data, with a fixed seed for reproducibility
)

model = ___()          # YOUR CODE — instantiate the linear regression model
model.fit(___, ___)    # YOUR CODE — fit on the training features and the training target

coef_series = pd.Series(___, index=___)                     # YOUR CODE — the fitted coefficients, labeled by feature name
coef_ranked = coef_series.abs().sort_values(ascending=___)  # YOUR CODE — sort so the largest-magnitude coefficient comes first

top_feature = coef_ranked.index[0]

print("Coefficients ranked by absolute value:")
for feat in coef_ranked.index:
    print(f"  {feat:<14} {coef_series[feat]:+.4f}")
print(f"\nHighest-weight feature: {top_feature}")

# --- checks ---
assert X_train.shape[0] + X_test.shape[0] == len(X), \
    "train + test should equal total samples"
assert hasattr(model, 'coef_'), \
    "model.coef_ is not available — did you call model.fit()?"
assert len(coef_series) == X.shape[1], \
    "coef_series should have one value per feature"
assert top_feature == 'AveBedrms', \
    "Sort descending by absolute value — check ascending=False"
print("✓ B1 complete!")

### B2 — Check for multicollinearity with VIF

The Variance Inflation Factor tells you how much a coefficient's variance is inflated by its correlation with the other features. `VIF > 5` signals a feature worth watching.

In [ ]:
# B2 — Compute Variance Inflation Factor for every feature
#
# add_constant is required — without it, VIF is computed as if the model has
# no intercept, which distorts the result.

X_const = sm.add_constant(X)

vif_df = pd.DataFrame({
    "Feature": X.columns,
    "VIF": [variance_inflation_factor(___.values, i) for i in range(1, ___.shape[1])],  # YOUR CODE — the constant-augmented feature matrix (needed for both its values and its column count)
})
vif_df = vif_df.sort_values("VIF", ascending=___).reset_index(drop=True)  # YOUR CODE — put the highest VIF first

flagged = vif_df[vif_df["VIF"] > ___]   # YOUR CODE — the VIF threshold named in the markdown above

print("Variance Inflation Factor — California Housing Features")
for _, row in vif_df.iterrows():
    flag = " <<" if row["VIF"] > 5 else ""
    print(f"  {row['Feature']:<14} {row['VIF']:>8.2f}{flag}")
print(f"\n{len(flagged)} feature(s) exceed the VIF > 5 threshold: {flagged['Feature'].tolist()}")

# --- checks ---
assert len(vif_df) == X.shape[1], \
    "vif_df should have one row per feature"
assert (vif_df["VIF"] > 0).all(), \
    "VIF values should all be positive"
assert len(flagged) == 4, \
    "Check your threshold — 4 features should exceed VIF > 5 for this dataset"
print("✓ B2 complete!")

### B3 — Watch polynomial regression overfit

A single feature transformed with `PolynomialFeatures` at increasing degree keeps improving on the training set. Fill in the blanks to compare train and test R² across degree.

In [ ]:
# B3 — Compare train vs. test R² as polynomial degree increases
#
# Using a small subsample (n=100) makes overfitting visible, the same way
# it was in the working session — the full dataset has too many rows for a
# single feature to overfit at any reasonable degree.

_idx = np.random.RandomState(42).choice(len(X), 100, replace=False)
x_small = X["MedInc"].values[_idx]
y_small = y.values[_idx]

x_train_s, x_test_s, y_train_s, y_test_s = train_test_split(
    x_small, y_small, test_size=0.2, random_state=42
)

degrees = [1, 2, 4, 8, 10]
train_scores, test_scores = [], []

for d in degrees:
    poly = PolynomialFeatures(degree=___, include_bias=False)   # YOUR CODE — the current loop's degree
    x_train_poly = poly.fit_transform(___.reshape(-1, 1))       # YOUR CODE — the training slice of the single feature
    x_test_poly = poly.transform(___.reshape(-1, 1))            # YOUR CODE — the test slice of the single feature

    poly_model = LinearRegression()
    poly_model.fit(___, ___)                                    # YOUR CODE — fit on the transformed training features and the training target

    train_scores.append(r2_score(y_train_s, poly_model.predict(x_train_poly)))
    test_scores.append(___(y_test_s, poly_model.predict(___)))  # YOUR CODE — the same scoring function used above, applied to predictions on the transformed test features

for d, tr, te in zip(degrees, train_scores, test_scores):
    print(f"Degree {d:>2}: train R² = {tr:6.3f}   test R² = {te:6.3f}")

# --- checks ---
assert len(train_scores) == len(degrees) == len(test_scores), \
    "Make sure the loop ran once per degree"
assert all(train_scores[i] <= train_scores[i + 1] for i in range(len(train_scores) - 1)), \
    "Training R² should never decrease as degree increases — check that you refit on x_train_poly each time"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A single train/test split gives one noisy read on overfitting. Cross-validation, plus a held-out test set that is never touched during model selection, gives a trustworthy answer to "what degree should I use?" Fill in the blanks below to select the best polynomial degree and evaluate it honestly.

In [ ]:
# Section C — Choosing polynomial degree with cross-validation

_idx_c = np.random.RandomState(42).choice(len(X), 100, replace=False)
x_c = X["MedInc"].values[_idx_c]
y_c = y.values[_idx_c]

# Held out once, untouched until the very end
x_pool, x_test_c, y_pool, y_test_c = train_test_split(
    ___, ___, test_size=___, random_state=___    # YOUR CODE — split the pooled feature and target arrays, holding out 20% with a fixed seed
)

degrees = list(range(1, 11))
kf = KFold(n_splits=___, shuffle=True, random_state=42)   # YOUR CODE — the fold count mentioned in this section's intro
cv_means = []

for d in degrees:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("sc", StandardScaler()),
        ("lr", ___()),                                                          # YOUR CODE — the base regression model used throughout this lab
    ])
    scores = cross_val_score(pipe, ___.reshape(-1, 1), ___, cv=kf, scoring="r2")  # YOUR CODE — the pooled feature and target arrays, not the held-out test set
    cv_means.append(scores.mean())

best_degree = degrees[int(np.argmax(cv_means))]

final_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=best_degree, include_bias=False)),
    ("sc", StandardScaler()),
    ("lr", LinearRegression()),
])
final_pipe.fit(x_pool.reshape(-1, 1), y_pool)
test_r2 = r2_score(y_test_c, final_pipe.predict(x_test_c.reshape(-1, 1)))

print(f"Best degree by mean 5-fold CV R²: {best_degree}  (CV R² = {cv_means[best_degree - 1]:.3f})")
print(f"Held-out test R² at degree {best_degree}: {test_r2:.3f}")

# Values below -1.5 are clipped for a readable plot; selection above used the real values.
_floor = -1.5
plot_means = [max(m, _floor) for m in cv_means]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(degrees, plot_means, marker='o')
ax.axvline(best_degree, linestyle='--', color='crimson', label=f"Selected degree = {best_degree}")
ax.set_xlabel("Polynomial Degree")
ax.set_ylabel("Mean 5-Fold CV R² (clipped at -1.5)")
ax.set_title("Choosing Polynomial Degree by Cross-Validation")
ax.legend()
plt.tight_layout()
plt.show()

# --- Section C checks ---
assert len(cv_means) == len(degrees), \
    "cv_means should have one value per degree"
assert best_degree == degrees[int(np.argmax(cv_means))], \
    "best_degree should be selected from cv_means, not chosen by hand"
assert 0.3 < test_r2 < 0.7, \
    f"test R² looks off ({test_r2:.3f}) — check the pipeline and the held-out split"
print("✓ Section C complete!")
print(f"  Cross-validation selected degree {best_degree} as the best-generalizing model.")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

B2 flags four features with VIF above 5. Does this mean you should immediately drop those features from the model? What would you check first before deciding?

*Your response here...*

**Question D2**

In Section C, cross-validation selected degree 1 as the best-generalizing model, even though a higher degree always fits the training data better. Why does "best on training data" and "best generalizing model" diverge, and why does that matter in practice?

*Your response here...*

**Question D3**

Look at the print output from B3 as degree increases. What would you expect to happen to the gap between train and test performance if you used a much larger dataset instead of a 100-row subsample? Why?

*Your response here...*